In [ ]:
# Install the modern Google ADK framework with GCP extensions and requests
!pip uninstall -y google-adk google-cloud-aiplatform
!pip install -q "google-adk[gcp]>=2.4.0" requests

print("✓ Dependencies installed successfully.")

In [ ]:
import os
import json
import requests
from typing import Dict, Any

# Import the modern Google Agent Development Kit framework
from google.adk import Agent

# Automatically extract Qwiklabs project parameters and enforce region mapping
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-04-3d3bb8d72ee3")
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-east4"  # Match your assigned global cluster location

print("✓ Core environment configurations loaded perfectly.")

In [ ]:
def get_current_weather(location: str) -> str:
    """Gets the live, real-time weather data for a specified USA city.

    Args:
        location: The name of the city and state (e.g., "Denver, CO").

    Returns:
        A JSON-formatted string detailing the temperature, conditions, and wind speed.
    """
    try:
        url = f"https://wttr.in/{location.replace(' ', '+')}?format=j1"
        response = requests.get(url, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            current = data['current_condition'][0]
            nearest_area = data['nearest_area'][0]
            
            weather_info = {
                "location": f"{nearest_area['areaName'][0]['value']}, {nearest_area['region'][0]['value']}, USA",
                "temperature_f": current['temp_F'],
                "feels_like_f": current['FeelsLikeF'],
                "condition": current['weatherDesc'][0]['value'],
                "humidity": current['humidity'],
                "wind_speed_mph": current['windspeedMiles']
            }
            return json.dumps(weather_info)
        else:
            return json.dumps({"error": f"City data unavailable (Status: {response.status_code})"})
    except Exception as e:
        return json.dumps({"error": str(e)})

print("✓ Weather tool verified and parsed successfully.")

In [ ]:
# Instantiating the weather companion agent using the ADK model controller pattern.
# Note: This removes the entire manual 'run_agent' tool extraction loop!
weather_agent = Agent(
    model="gemini-2.5-flash",
    name="adk_weather_agent",
    instruction=(
        "You are a helpful weather assistant. Use the tools provided to look up real-time data "
        "and provide a friendly, concise weather summary including temperature, conditions, and any notable details."
    ),
    tools=[get_current_weather]
)

print("✓ ADK Agent created with tool alignment mappings completed.")

In [ ]:
test_cities = [
    "New York City, NY",
    "Denver, CO",
    "San Francisco, CA",
    "Atlanta, GA"
]

print("============================================================")
print("STARTING GOOGLE ADK WEATHER AGENT TEST SUITE")
print("============================================================\n")

for city in test_cities:
    prompt = f"Can you give me a quick weather breakdown for {city}?"
    print(f"User: {prompt}")
    try:
        # The ADK automatically identifies that it needs the get_current_weather tool,
        # executes it behind the scenes, feeds it back to the model, and outputs text.
        summary = weather_agent.run(prompt)
        print(f"Agent Summary:\n{summary}")
    except Exception as e:
        print(f"Execution Error: {e}")
    print("-" * 60)